# Day D6 — 연결성: seed 상관, PPI, ICA, 유효 연결성, 휴지기 네트워크

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/D-fmri/day-D6.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/D-fmri/day-D6.md)

In [ ]:
# Colab 기본 런타임에 없는 패키지 설치
%pip install -q nilearn

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 과제가 만든 가짜 상관과 잔차 상관

In [ ]:
import numpy as np
import statsmodels.api as sm
from scipy.stats import gamma

rng = np.random.default_rng(22)
TR, n, dt = 2.0, 200, 0.1
t_hrf = np.arange(0, 32, dt)
hrf = gamma.pdf(t_hrf, 6) - gamma.pdf(t_hrf, 16) / 6
t_hi = np.arange(0, n * TR, dt)
box = ((t_hi % 40) < 20).astype(float)
task = (np.convolve(box, hrf)[: t_hi.size] * dt)[:: int(TR / dt)]
task /= task.max()

# 세 영역: A와 B는 서로 영향을 주지 않지만 둘 다 과제에 반응한다. C는 A와 실제로 이어져 있다
a_own = rng.normal(0, 1, n)
A = 2.0 * task + a_own
B = 2.0 * task + rng.normal(0, 1, n)
C = 2.0 * task + 0.6 * a_own + rng.normal(0, 0.8, n)

X = sm.add_constant(task)                       # '방해 모형': 과제 효과(필요하면 움직임 등도)
res = {k: sm.OLS(v, X).fit().resid for k, v in {"A": A, "B": B, "C": C}.items()}

print("원 시계열 상관   A–B: %.2f   A–C: %.2f" % (np.corrcoef(A, B)[0, 1], np.corrcoef(A, C)[0, 1]))
print("잔차 상관        A–B: %.2f   A–C: %.2f" % (np.corrcoef(res["A"], res["B"])[0, 1],
                                                 np.corrcoef(res["A"], res["C"])[0, 1]))

### 예시 2 — PPI: 조건에 따라 달라지는 결합 찾기

In [ ]:
import numpy as np
import statsmodels.api as sm
from scipy.stats import gamma

rng = np.random.default_rng(31)
TR, n, dt = 2.0, 240, 0.1
t_hrf = np.arange(0, 32, dt)
hrf = gamma.pdf(t_hrf, 6) - gamma.pdf(t_hrf, 16) / 6
t_hi = np.arange(0, n * TR, dt)

def conv(x_hi):
    r = (np.convolve(x_hi, hrf)[: t_hi.size] * dt)[:: int(TR / dt)]
    return r

# 두 조건이 번갈아 나오는 블록 설계: 조건 1(+1)과 조건 2(−1)
cond_hi = np.where((t_hi % 60) < 30, 1.0, -1.0)
psych = conv(cond_hi)                               # '심리' 회귀변수(과제 차이)

# 가상 '신경 활동': seed 영역의 자발적 요동. target은 조건 1일 때만 seed를 따라간다
seed_neural_hi = np.repeat(rng.normal(0, 1, n), int(TR / dt))
coupling_hi = np.where(cond_hi > 0, 0.8, 0.0)
seed = conv(seed_neural_hi) + rng.normal(0, 0.3, n)
target = conv(coupling_hi * seed_neural_hi) + 0.5 * psych + rng.normal(0, 0.3, n)

# PPI 회귀변수: 학습용으로 '신경 수준'의 seed × 조건을 곱한 뒤 HRF를 다시 씌운다.
# (실제 seed의 신경 신호는 모르므로 SPM 등은 BOLD에서 deconvolution으로 추정한다)
ppi = conv(seed_neural_hi * cond_hi)
naive_ppi = (seed - seed.mean()) * np.sign(psych)   # 흔한 지름길: BOLD끼리 그냥 곱하기

for name, p in [("deconvolution형 PPI", ppi), ("BOLD 곱 PPI(지름길)", naive_ppi)]:
    X = sm.add_constant(np.column_stack([psych, seed, p]))   # 과제, seed, 상호작용을 모두 넣는다
    fit = sm.OLS(target, X).fit()
    print(f"{name:20} PPI t = {fit.tvalues[3]:5.2f} | 과제 t = {fit.tvalues[1]:5.2f}"
          f" | PPI–과제 상관 {np.corrcoef(p, psych)[0, 1]:5.2f}")

### 예시 3 — 휴지기 연결성 행렬과 움직임

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(41)
n, names = 300, ["DMN_1", "DMN_2", "DMN_3", "MOT_1", "MOT_2", "MOT_3"]
# 가상 휴지기 자료: 두 '네트워크'가 각자 공통 요동을 갖는다
net_dmn, net_mot = rng.normal(size=n), rng.normal(size=n)
ts = np.column_stack([0.7 * net_dmn + rng.normal(size=n) for _ in range(3)] +
                     [0.7 * net_mot + rng.normal(size=n) for _ in range(3)])

# 머리 움직임이 모든 영역에 같은 흔적을 남긴다고 가정 (실제로는 영역마다 다르다)
fd_like = np.abs(rng.normal(0, 0.1, n)); fd_like[[50, 51, 120, 180, 181, 182, 250]] += 1.5
dirty = ts + 3.0 * fd_like[:, None]

def corr_table(x):
    return pd.DataFrame(np.corrcoef(x.T), index=names, columns=names).round(2)

def regress_out(x, nuis):
    Z = np.column_stack([np.ones(n), nuis])
    beta, *_ = np.linalg.lstsq(Z, x, rcond=None)
    return x - Z @ beta

clean = regress_out(dirty, fd_like)
for label, x in [("움직임 섞인 자료", dirty), ("움직임 회귀 뒤", clean)]:
    r = np.corrcoef(x.T)
    between = r[:3, 3:].mean()
    within = (r[:3, :3][np.triu_indices(3, 1)].mean() + r[3:, 3:][np.triu_indices(3, 1)].mean()) / 2
    print(f"{label:10} 네트워크 안 평균 r {within:.2f} | 네트워크 사이 평균 r {between:.2f}")

# 그래프로 요약: |r| > 0.2를 연결(edge)로 보고 마디(node)마다 연결 수(degree)를 센다
for label, x in [("움직임 섞인 자료", dirty), ("움직임 회귀 뒤", clean)]:
    r = np.corrcoef(x.T); np.fill_diagonal(r, 0)
    adj = np.abs(r) > 0.2
    print(f"{label:10} 연결 수 {adj.sum() // 2:2d} | degree {adj.sum(1).tolist()}")
print(corr_table(clean))

연결 행렬을 그림으로 남기려면 예시 3 뒤에 아래를 붙입니다(그림 파일만 저장).

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

Path("figures/advanced/D").mkdir(parents=True, exist_ok=True)
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, (label, x) in zip(axes, [("with motion", dirty), ("motion regressed", clean)]):
    im = ax.imshow(np.corrcoef(x.T), vmin=-1, vmax=1, cmap="RdBu_r")
    ax.set_xticks(range(6)); ax.set_xticklabels(names, rotation=90)
    ax.set_yticks(range(6)); ax.set_yticklabels(names); ax.set_title(label)
fig.colorbar(im, ax=axes, shrink=0.8)
fig.savefig("figures/advanced/D/D6_connectivity_matrix.png", dpi=150, bbox_inches="tight")

### 참고 — 실제 자료에서 ROI 시계열과 연결 행렬을 얻는 형태 (nilearn 필요)

> 이 코드는 `pip install nibabel nilearn`으로 설치한 환경(nibabel 5.5, nilearn 0.14.1)에서, 같은 이름의 **가상 파일**을 만들어 그대로 실행해 오류 없이 도는 것을 확인했습니다. 출력은 여러분의 실제 파일에 따라 달라지므로 싣지 않았습니다.
> 첫 실행 때 Harvard-Oxford 아틀라스(약 26 MB)를 홈 폴더의 `nilearn_data/`에 내려받습니다.

In [ ]:
# nilearn이 필요하다: pip install nilearn
from nilearn import datasets
from nilearn.maskers import NiftiLabelsMasker
from nilearn.connectome import ConnectivityMeasure
atlas = datasets.fetch_atlas_harvard_oxford("cort-maxprob-thr25-2mm")
masker = NiftiLabelsMasker(labels_img=atlas.maps, standardize=True, t_r=2.0,
                           high_pass=0.01, low_pass=0.1)
ts = masker.fit_transform("sub-01_task-rest_bold.nii.gz", confounds=confounds_df)  # 방해 신호를 함께 회귀
r = ConnectivityMeasure(kind="correlation").fit_transform([ts])[0]